# 623 SPP v19: routed/page-local LSTM + exact learned action grammar

The external input is byte-for-byte the validated v18 package: chronological `DEMAND(addr)` and `CACHE_FILL(evicted_addr)` only. v19 routes those callbacks through separate recurrent paths, maintains causal page-local state with learned age-conditioned validity, and learns a rank-wise keyed `STOP/EMIT` grammar. An emitted action uses an autoregressive exact signed ZigZag + canonical LEB128 increment; `delta=0` is legal, only duplicate targets within that callback are probability-masked, fill is predicted after the actual hard target, and that hard action is recurrent feedback. Teacher prefixes and targets exist only in separate output-likelihood factors and never mutate the sampled rollout. There is no threshold, hurdle, Poisson count, GMM, candidate bank, page restriction, policy degree cap, or nearest-delta fallback. A 52-rank sampler-precision watchdog fails the whole run without replay instead of truncating. This remains matched-input open-loop replay.

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select a GPU runtime (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
REPO='/content/cache_arch'; TOKEN=userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN to Colab Secrets'
ASKPASS='/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$GITHUB_TOKEN" ;; esac\n'); os.chmod(ASKPASS,0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':ASKPASS,'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN':TOKEN})
try:
 if not os.path.isdir(REPO): subprocess.run(['git','clone','https://github.com/Angelawoo572/cache_arch.git',REPO],check=True,env=env)
 else: subprocess.run(['git','-C',REPO,'pull','--ff-only','origin','main'],check=True,env=env)
finally: pathlib.Path(ASKPASS).unlink(missing_ok=True)
print(torch.cuda.get_device_name(0),subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
SCRIPT=f'{REPO}/formal_NN_training/experiments/623_offline_lstm_spp/python/train_and_offline_infer.py'
MODEL_CONTRACT=json.loads(subprocess.check_output([sys.executable,SCRIPT,'--describe-model-points'],text=True))
RUN_ID=MODEL_CONTRACT['run_id']
DRIVE_ROOT=f'/content/drive/MyDrive/cache_prefetch_623_spp/{RUN_ID}'
INPUT_DIR=f'{DRIVE_ROOT}/colab_input'; OUTPUT_ROOT=f'{DRIVE_ROOT}/colab_output'; os.makedirs(DRIVE_ROOT,exist_ok=True)
name=f'{RUN_ID}.colab_input.tar.gz'; uploaded=files.upload(); assert name in uploaded,f'Select {name}'
archive=f'{DRIVE_ROOT}/{name}'; pathlib.Path(archive).write_bytes(uploaded[name])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR,exist_ok=True)
with tarfile.open(archive,'r:gz') as handle: handle.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
 expected,item=record.split(maxsplit=1); item=item.lstrip('*')
 assert hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{item}').read_bytes()).hexdigest()==expected
print('verified byte-identical reused input',archive)

In [ ]:
TRACE=MODEL_CONTRACT['trace']; POLICY=MODEL_CONTRACT['policy']; ROLES=('train','guard','eval')
INPUTS={role:{'stream':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz','teacher':f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_teacher_actions.csv.gz'} for role in ROLES}
for items in INPUTS.values():
 for path in items.values(): assert os.path.isfile(path),path
historical_manifest=json.loads(pathlib.Path(f'{INPUT_DIR}/collection_manifest.json').read_text())
expected_input={'status':'PASS','experiment_revision':MODEL_CONTRACT['experiment_revision'],'event_logger_schema':'623_causal_trigger_fill_v6','source_decision_effective_external_input':MODEL_CONTRACT['external_input_fields'],'model_input_is_causal_external_event_sequence_only':True,'cache_fill_feedback_used_as_raw_external_input':True,'same_external_input_contract':True,'training_inference_input_encoder_identical':True,'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_candidates_used_as_model_inputs':False,'normal_policy_private_state_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,'normal_policy_request_rate_used_as_budget':False,'probability_threshold_used':False,'neural_degree_cap':None}
bad={k:(historical_manifest.get(k),v) for k,v in expected_input.items() if historical_manifest.get(k)!=v}; assert not bad,bad
assert historical_manifest['training_runtime_fields']==MODEL_CONTRACT['external_input_fields']==historical_manifest['inference_runtime_fields']
SOURCE=f'{INPUT_DIR}/spp_source_contract.json'
COLLECTION_MANIFEST_ROLE='historical_input_package_provenance_only'

In [ ]:
LOCAL_OUTPUT=f'/content/{RUN_ID}_colab_output'
if os.path.isdir(LOCAL_OUTPUT): shutil.rmtree(LOCAL_OUTPUT)
os.makedirs(LOCAL_OUTPUT)
SPECS=[{'tag':point['tag'],'family':point['family'],'size':point['size'],'pair':point['pair_id'],'parameters':point['parameter_count']} for point in MODEL_CONTRACT['points']]
assert [spec['size'] for spec in SPECS]==[8,16] and all(spec['parameters']<10000 for spec in SPECS),SPECS
SWEEP=[]
for spec in SPECS:
 out=f"{LOCAL_OUTPUT}/{spec['tag']}"; cmd=[sys.executable,SCRIPT,'--policy',POLICY]
 for role in ROLES: cmd += [f'--{role}-stream',INPUTS[role]['stream'],f'--{role}-teacher-actions',INPUTS[role]['teacher']]
 cmd += ['--source-contract',SOURCE,'--out-dir',out,'--model-family',spec['family'],'--model-size',str(spec['size']),'--pair-id',spec['pair'],'--device','cuda','--seed','7','--decoder-seed','7','--epochs','10','--chunk-len','1024','--accumulate-chunks','16']
 print('\nTraining',spec['tag'],' '.join(cmd),flush=True); subprocess.run(cmd,check=True)
 meta=json.loads(pathlib.Path(f'{out}/run_metadata.json').read_text())
 expected={
  'run_id':RUN_ID,'model_tag':spec['tag'],'model_family':'lstm','track_model_family':'lstm',
  'operation':MODEL_CONTRACT['operation'],'model_revision':MODEL_CONTRACT['model_revision'],'decoder_revision':MODEL_CONTRACT['decoder_revision'],
  'parameter_count':spec['parameters'],'runtime_feature_count':MODEL_CONTRACT['runtime_feature_count'],'matched_normal_prefetcher':POLICY,
  'same_external_input_contract':True,'training_inference_input_encoder_identical':True,
  'decoder_training_mode':'sampled_rank_grammar_rollout_with_separate_teacher_prefix_output_nll',
  'decoder_previous_teacher_action_used_as_input':True,'decoder_previous_teacher_action_used_as_input_scope':'isolated_loss_only_teacher_prefix_likelihood_branch','decoder_previous_teacher_action_used_as_main_rollout_input':False,'decoder_free_running_self_test':'PASS',
  'teacher_prefix_advances_loss_only_likelihood_byte_state':True,'teacher_prefix_used_as_main_rollout_recurrent_feedback':False,'teacher_action_values_used_as_main_rollout_recurrent_feedback':False,
  'model_input_is_causal_external_event_sequence_only':True,'cache_fill_feedback_used_as_raw_external_input':True,'model_does_not_use_pc':True,
  'normal_policy_outputs_used_as_model_inputs':False,'normal_policy_outputs_used_as_training_targets':True,
  'probability_threshold_used':False,'neural_degree_cap':None,'gate_training_objective':None,'gate_decoding_rule':None,
  'request_count_training_objective':'rankwise_unweighted_stop_emit_categorical_nll',
  'request_count_decoding_rule':'first_keyed_learned_STOP_token_ends_action_sequence','request_count_sampling_performed':True,
  'stop_emit_sampling_rule':'event_rank_keyed_categorical_inverse_cdf','stop_emit_sampler_representability_check':'STOP_mass_strictly_above_open_uniform_half_bin','action_rollout_fail_closed_watchdog_ranks':MODEL_CONTRACT['action_rollout_watchdog_ranks'],'action_rollout_watchdog_is_neural_degree_cap':False,
  'delta_mixture_components':0,'delta_training_objective':'exact_autoregressive_teacher_prefix_canonical_leb128_nll_with_sampled_history_duplicate_support',
  'delta_decoding_rule':'keyed_exact_signed_zigzag_canonical_leb128','delta_zero_allowed':True,'self_target_actions_allowed':True,
  'duplicate_target_handling':'mask_categorical_probability_and_renormalize','duplicate_prefix_feasibility_mask_used':True,'delta_legality_fallback':None,
  'fill_training_objective':'unweighted_two_class_cross_entropy_conditioned_on_teacher_target_loss_only',
  'fill_conditioned_on_actual_emitted_target':True,'fill_argmax_used':False,
  'routed_demand_fill_recurrent_paths':True,'page_local_causal_state':True,
  'common_random_numbers_across_capacities':True,'strict_common_random_numbers_across_capacities':True,
  'cross_event_rng_state_used':False,'stochastic_decoding_reproducible':True,
  'same_source_input_offline_claim_allowed':True,'closed_loop_live_claim_allowed':False,
  'keyed_sampling_self_test':'PASS','rank_stop_emit_grammar_self_test':'PASS','exact_leb128_codec_self_test':'PASS','duplicate_prefix_no_dead_end_self_test':'PASS','teacher_prefix_state_isolation_self_test':'PASS','stop_sampler_representability_self_test':'PASS','always_emit_watchdog_self_test':'PASS','integer_csv_exactness_self_test':'PASS',
  'target_conditioned_fill_self_test':'PASS','routed_page_state_self_test':'PASS',
  'experiment_revision':MODEL_CONTRACT['experiment_revision']
 }
 bad={k:(meta.get(k),v) for k,v in expected.items() if meta.get(k)!=v}; assert not bad,bad
 assert meta['model_point_contract']==MODEL_CONTRACT and meta['parameter_count']<10000
 assert meta['training_runtime_fields']==MODEL_CONTRACT['external_input_fields']==meta['inference_runtime_fields']
 hashes={meta.get('runtime_encoder_sha256'),meta.get('training_runtime_encoder_sha256'),meta.get('inference_runtime_encoder_sha256')}
 assert len(hashes)==1 and len(next(iter(hashes)))==64,hashes
 fills=meta['offline_nn_fill_level_counts']; assert sum(fills.values())==meta['offline_nn_entries']
 assert meta['offline_nn_entries']==meta['materialized_distinct_action_count']==meta['raw_predicted_action_count']
 assert meta['action_legality_diagnostics']['duplicate_target_actions']==0
 assert meta['peak_persistent_recurrent_state_bytes']>0 and meta['dynamic_page_state_pages']>0
 SWEEP.append({k:meta[k] for k in ('model_tag','model_family','model_size','architecture_pair_id','parameter_count','parameter_storage_bytes_float32','peak_persistent_recurrent_state_bytes','decision_rule','offline_normal_entries','offline_nn_entries','offline_nn_fill_level_counts','heldout_behavior_metrics')})
if os.path.isdir(OUTPUT_ROOT): shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(LOCAL_OUTPUT,OUTPUT_ROOT)
pathlib.Path(f'{OUTPUT_ROOT}/sweep_manifest.json').write_text(json.dumps({'trace':TRACE,'input_revision':MODEL_CONTRACT['experiment_revision'],'model_revision':MODEL_CONTRACT['model_revision'],'collection_manifest_role':COLLECTION_MANIFEST_ROLE,'points':SWEEP},indent=2)+'\n')
print(json.dumps(SWEEP,indent=2))

In [ ]:
OUTPUT_ARCHIVE=f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(OUTPUT_ARCHIVE,'w:gz') as archive:
 for item in pathlib.Path(OUTPUT_ROOT).iterdir(): archive.add(item,arcname=item.name)
print('DONE',OUTPUT_ARCHIVE,os.path.getsize(OUTPUT_ARCHIVE),'bytes')
files.download(OUTPUT_ARCHIVE)

Copy the v19 output archive to the matching Sacramento run and launch replay. The input remains the byte-identical v18 capture; teacher actions are labels/comparator data only, and the comparison remains open-loop.